# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/clever-dhruv/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [31]:
from huggingface_hub import login

login(token=HF_TOKEN)
print("Hugging Face connected")

Hugging Face connected


In [32]:
from huggingface_hub import list_repo_files

files = list(list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
))

print("\n".join(files[:30]))

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [33]:
from huggingface_hub import hf_hub_download
import pandas as pd

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

df = pd.read_parquet(file_path)

print(df.columns.tolist())
print(df.shape)

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
(9841378, 30)


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""One row represents one client’s content item on one report date. I will use March 2026 as the development month."""
# Verify the grain for March 2026

total_rows = len(df)

unique_grain = df[
    ["report_date", "client_hash_id", "content_hash_id"]
].drop_duplicates().shape[0]

print("Total rows:", total_rows)
print("Unique report_date + client + content combinations:", unique_grain)
print("Grain matches:", total_rows == unique_grain)

Total rows: 9841378
Unique report_date + client + content combinations: 9841378
Grain matches: True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""Features:
- gsc_impressions — historical search visibility available at the decision moment.
- gsc_clicks — historical search clicks available at the decision moment.
- gsc_avg_position — historical search ranking available at the decision moment.
- ga4_pageviews — historical page traffic available at the decision moment.
- ga4_engaged_sessions — historical engagement available at the decision moment.

Label:
- Label / proxy:
- refresh_proxy — a directional proxy equal to 1 when April clicks are lower than March clicks, and 0 otherwise. It is used as a future outcome for the exercise, not as proof that a page requires a refresh.

Context:
- report_date — identifies when the observation was recorded.
- client_hash_id — identifies the client without exposing the client name.
- content_hash_id — identifies the content item without exposing its URL.

Excluded:
- April/future performance fields are excluded from the honest feature set because they are unavailable at the March decision moment and would cause leakage. """

'Features:\n- gsc_impressions — historical search visibility available at the decision moment.\n- gsc_clicks — historical search clicks available at the decision moment.\n- gsc_avg_position — historical search ranking available at the decision moment.\n- ga4_pageviews — historical page traffic available at the decision moment.\n- ga4_engaged_sessions — historical engagement available at the decision moment.\n\nLabel:\n- Label / proxy:\n- refresh_proxy — a directional proxy equal to 1 when April clicks are lower than March clicks, and 0 otherwise. It is used as a future outcome for the exercise, not as proof that a page requires a refresh.\n\nContext:\n- report_date — identifies when the observation was recorded.\n- client_hash_id — identifies the client without exposing the client name.\n- content_hash_id — identifies the content item without exposing its URL.\n\nExcluded:\n- April/future performance fields are excluded from the honest feature set because they are unavailable at the Ma

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Row count:", len(df))
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())

Row count: 9841378
Date range: 2026-03-01 to 2026-03-31


In [37]:
availability_query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available
FROM df
"""

print(con.execute(availability_query).fetchdf())

   total_rows  gsc_available  ga4_available
0     9841378        3611061         413966


In [38]:
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

feature_df = df[
    ["report_date", "client_hash_id", "content_hash_id"] + features
].copy()

print(feature_df.shape)
display(feature_df.head())

(9841378, 8)


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,NaN,NaN


In [39]:
"""Five-feature frame:

- gsc_impressions — available when the decision is made because it is historical search performance for the observation period.
- gsc_clicks — available when the decision is made because it is historical search traffic for the observation period.
- gsc_avg_position — available when the decision is made because it is historical search ranking for the observation period.
- ga4_pageviews — available when the decision is made because it is historical page traffic for the observation period.
- ga4_engaged_sessions — available when the decision is made because it is historical engagement for the observation period."""

'Five-feature frame:\n\n- gsc_impressions — available when the decision is made because it is historical search performance for the observation period.\n- gsc_clicks — available when the decision is made because it is historical search traffic for the observation period.\n- gsc_avg_position — available when the decision is made because it is historical search ranking for the observation period.\n- ga4_pageviews — available when the decision is made because it is historical page traffic for the observation period.\n- ga4_engaged_sessions — available when the decision is made because it is historical engagement for the observation period.'

In [40]:
print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [41]:
april_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

april_df = pd.read_parquet(april_path)

print(april_df.shape)

(10424730, 30)


In [42]:
march = df.groupby("content_hash_id", as_index=False).agg(
    march_clicks=("gsc_clicks", "sum"),
    march_impressions=("gsc_impressions", "sum")
)

april = april_df.groupby("content_hash_id", as_index=False).agg(
    april_clicks=("gsc_clicks", "sum"),
    april_impressions=("gsc_impressions", "sum")
)

future = march.merge(
    april,
    on="content_hash_id",
    how="inner"
)

print(future.shape)
display(future.head())

(331436, 5)


,content_hash_id,march_clicks,march_impressions,april_clicks,april_impressions
0,content_000005d4ced12088,0,86,0,81
1,content_00001e488b74b799,0,0,0,0
2,content_00007bd2985b77c3,0,47,0,48
3,content_00008950670cb6b5,0,0,0,0
4,content_0000a348850eb1fc,0,0,0,0


In [43]:
future["click_change"] = future["april_clicks"] - future["march_clicks"]

future["refresh_proxy"] = (
    future["click_change"] < 0
).astype(int)

print(future["refresh_proxy"].value_counts())

refresh_proxy
0    286334
1     45102
Name: count, dtype: int64


In [44]:
leaky_features = future[[
    "march_clicks",
    "march_impressions",
    "refresh_proxy"
]].copy()

print(leaky_features.head())

   march_clicks  march_impressions  refresh_proxy
0             0                 86              0
1             0                  0              0
2             0                 47              0
3             0                  0              0
4             0                  0              0


In [45]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X = leaky_features[["march_clicks", "march_impressions", "refresh_proxy"]]
y = leaky_features["refresh_proxy"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy with leaked feature:", accuracy_score(y_test, pred))

Accuracy with leaked feature: 1.0


In [46]:
honest_features = future[
    ["march_clicks", "march_impressions"]
].copy()

X = honest_features
y = future["refresh_proxy"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = DecisionTreeClassifier(
    random_state=42,
    max_depth=5
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Honest accuracy:", accuracy_score(y_test, pred))

Honest accuracy: 0.9319032102341298


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [47]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
""" This slice cannot tell us the true business reason why a page's performance changed. A drop in clicks may come from seasonality, search-engine changes, competition, or other factors that are not captured in this data. The refresh proxy is therefore directional decision-support, not proof that a page needs a content refresh."""

" This slice cannot tell us the true business reason why a page's performance changed. A drop in clicks may come from seasonality, search-engine changes, competition, or other factors that are not captured in this data. The refresh proxy is therefore directional decision-support, not proof that a page needs a content refresh."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [48]:
import duckdb

con = duckdb.connect()

In [49]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", bool(HF_TOKEN))

HF token loaded: True
